In [1]:
# Notebook: 08 - Save Model Professionally

import pandas as pd
import numpy as np
import pickle
import json
import os
import hashlib
import datetime
from sklearn.metrics         import r2_score, mean_squared_error
from sklearn.preprocessing   import StandardScaler

print("Libraries loaded.")
print(f"Working directory: {os.getcwd()}")
print(f"Timestamp: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Libraries loaded.
Working directory: C:\Users\hp\Desktop\used_car_price_prediction\notebooks
Timestamp: 2026-05-24 20:09:33


In [2]:
print("""
A trained ML model in production needs MORE than just
the model weights. It needs a complete package:

PROBLEM: A model trained on log-transformed, encoded data
         cannot predict on raw input from a user.

The app receives:    Present_Price=8.0, Kms=45000, Year=2018
The model expects:   Present_Price_log, Kms_Driven_log,
                     Car_Age, Fuel_Diesel, Fuel_Petrol ...

Without saving the SCALER and ENCODERS, you cannot
transform new user input the same way training data
was transformed. Your predictions will be garbage.

WHAT WE SAVE:
  1. champion_model.pkl     — the trained model
  2. scaler.pkl             — fitted StandardScaler
  3. feature_columns.pkl    — exact column order
  4. champion_meta.json     — metrics, version, timestamp
  5. predict_pipeline.pkl   — complete prediction function

WHY COLUMN ORDER MATTERS:
  If training used: [Present_Price_log, Car_Age, Kms_log ...]
  But prediction sends: [Car_Age, Present_Price_log, Kms_log ...]
  Every prediction will be SILENTLY WRONG.
  Saving the column order prevents this.
""")


A trained ML model in production needs MORE than just
the model weights. It needs a complete package:

PROBLEM: A model trained on log-transformed, encoded data
         cannot predict on raw input from a user.

The app receives:    Present_Price=8.0, Kms=45000, Year=2018
The model expects:   Present_Price_log, Kms_Driven_log,
                     Car_Age, Fuel_Diesel, Fuel_Petrol ...

Without saving the SCALER and ENCODERS, you cannot
transform new user input the same way training data
was transformed. Your predictions will be garbage.

WHAT WE SAVE:
  1. champion_model.pkl     — the trained model
  2. scaler.pkl             — fitted StandardScaler
  3. feature_columns.pkl    — exact column order
  4. champion_meta.json     — metrics, version, timestamp
  5. predict_pipeline.pkl   — complete prediction function

WHY COLUMN ORDER MATTERS:
  If training used: [Present_Price_log, Car_Age, Kms_log ...]
  But prediction sends: [Car_Age, Present_Price_log, Kms_log ...]
  Every prediction w

In [3]:
# LOAD ALL COMPONENTS


# Load the champion model
with open('../models/champion_model.pkl', 'rb') as f:
    champion_model = pickle.load(f)

# Load the scaler
with open('../models/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

# Load feature columns
with open('../models/feature_columns.pkl', 'rb') as f:
    feature_columns = pickle.load(f)

# Load champion metadata
with open('../models/champion_meta.json', 'r') as f:
    champion_meta = json.load(f)

# Load test data to verify everything still works
X_test_raw = pd.read_csv('../data/processed/X_test_unscaled.csv')
X_test     = pd.read_csv('../data/processed/X_test.csv')
y_test     = pd.read_csv('../data/processed/y_test.csv').squeeze()

print("All components loaded:")
print(f"  Model type:      {type(champion_model).__name__}")
print(f"  Champion name:   {champion_meta['champion_name']}")
print(f"  Features:        {feature_columns}")
print(f"  Needs scaling:   {champion_meta['needs_scaling']}")
print(f"  Saved test R²:   {champion_meta['test_r2']}")

# Quick sanity check — does the loaded model still predict?
needs_scaling = champion_meta['needs_scaling']
X_verify = X_test if needs_scaling else X_test_raw
preds    = champion_model.predict(X_verify)
r2_check = r2_score(y_test, preds)
print(f"\nLoaded model R² check: {r2_check:.4f}")
print(f"Matches saved R²:      "
      f"{'✅ YES' if abs(r2_check - champion_meta['test_r2']) < 0.001 else '❌ MISMATCH'}")

All components loaded:
  Model type:      Ridge
  Champion name:   Ridge Regression
  Features:        ['Present_Price_log', 'Car_Age', 'Kms_Driven_log', 'Fuel_Diesel', 'Fuel_Petrol', 'Transmission_enc', 'Seller_Type_enc', 'Owner']
  Needs scaling:   True
  Saved test R²:   0.9745

Loaded model R² check: 0.9745
Matches saved R²:      ✅ YES


In [4]:
# ============================================================
# BUILD A COMPLETE PREDICTION PIPELINE
# ============================================================
# This is the most important cell in this notebook.
#
# We build a single function that:
#   1. Takes raw user input (exactly what a user types)
#   2. Applies ALL the same transformations as training
#   3. Returns a prediction in actual Rupees Lakhs
#
# This function is EVERYTHING the Streamlit app needs.
# It encapsulates the entire feature engineering pipeline.

def predict_car_price(
    present_price,   # current showroom price in Lakhs
    kms_driven,      # total kilometres driven
    fuel_type,       # 'Petrol', 'Diesel', or 'CNG'
    seller_type,     # 'Dealer' or 'Individual'
    transmission,    # 'Manual' or 'Automatic'
    owner,           # number of previous owners (0,1,2,3)
    year,            # year of manufacture
    model,           # the trained ML model
    scaler,          # the fitted StandardScaler
    feature_cols,    # exact list of feature column names
    needs_scaling,   # True for linear models, False for trees
    current_year=2024
):
    """
    Predict the selling price of a used car from raw inputs.

    Parameters — all exactly as a user would enter them.
    Returns    — predicted price in Lakhs (₹).
    """

    # ── Step 1: Engineer features (same as training) ──────────
    car_age       = current_year - year
    present_price_log = np.log1p(present_price)
    kms_driven_log    = np.log1p(kms_driven)

    # ── Step 2: Encode categorical variables ──────────────────
    # One-hot: Fuel_Type
    fuel_diesel = 1 if fuel_type == 'Diesel' else 0
    fuel_petrol = 1 if fuel_type == 'Petrol' else 0
    # CNG = Fuel_Diesel=0 AND Fuel_Petrol=0

    # Label encode: Seller_Type
    seller_type_enc = 1 if seller_type == 'Dealer' else 0

    # Label encode: Transmission
    transmission_enc = 1 if transmission == 'Manual' else 0

    # ── Step 3: Build the feature row in exact column order ───
    feature_dict = {
        'Present_Price_log': present_price_log,
        'Kms_Driven_log':    kms_driven_log,
        'Car_Age':           car_age,
        'Owner':             owner,
        'Fuel_Diesel':       fuel_diesel,
        'Fuel_Petrol':       fuel_petrol,
        'Seller_Type_enc':   seller_type_enc,
        'Transmission_enc':  transmission_enc,
    }

    # Create DataFrame with exact column order from training
    input_df = pd.DataFrame([feature_dict])[feature_cols]

    # ── Step 4: Scale if needed (linear models only) ──────────
    if needs_scaling:
        input_scaled = scaler.transform(input_df)
        input_final  = pd.DataFrame(input_scaled,
                                    columns=feature_cols)
    else:
        input_final = input_df

    # ── Step 5: Predict and reverse log transform ─────────────
    log_prediction      = model.predict(input_final)[0]
    price_in_lakhs      = np.expm1(log_prediction)

    return round(price_in_lakhs, 2)


# ── Test the pipeline with a known car ────────────────────────
print("Testing prediction pipeline...\n")

test_cases = [
    {
        'name':          'Honda City 2017 Petrol',
        'present_price': 9.85,
        'kms_driven':    35000,
        'fuel_type':     'Petrol',
        'seller_type':   'Dealer',
        'transmission':  'Manual',
        'owner':         0,
        'year':          2017,
    },
    {
        'name':          'Toyota Innova 2015 Diesel',
        'present_price': 15.5,
        'kms_driven':    75000,
        'fuel_type':     'Diesel',
        'seller_type':   'Dealer',
        'transmission':  'Manual',
        'owner':         1,
        'year':          2015,
    },
    {
        'name':          'Maruti Swift 2019 Petrol Auto',
        'present_price': 6.5,
        'kms_driven':    20000,
        'fuel_type':     'Petrol',
        'seller_type':   'Individual',
        'transmission':  'Automatic',
        'owner':         0,
        'year':          2019,
    },
]

print(f"{'Car':<35} {'Predicted Price':>16}")
print("─" * 55)
for tc in test_cases:
    price = predict_car_price(
        present_price = tc['present_price'],
        kms_driven    = tc['kms_driven'],
        fuel_type     = tc['fuel_type'],
        seller_type   = tc['seller_type'],
        transmission  = tc['transmission'],
        owner         = tc['owner'],
        year          = tc['year'],
        model         = champion_model,
        scaler        = scaler,
        feature_cols  = feature_columns,
        needs_scaling = needs_scaling,
    )
    print(f"{tc['name']:<35} ₹{price:>8.2f} Lakhs")

print("\nPipeline working correctly!")

Testing prediction pipeline...

Car                                  Predicted Price
───────────────────────────────────────────────────────
Honda City 2017 Petrol              ₹    6.97 Lakhs
Toyota Innova 2015 Diesel           ₹    9.69 Lakhs
Maruti Swift 2019 Petrol Auto       ₹    7.43 Lakhs

Pipeline working correctly!


In [5]:
# ============================================================
# VALIDATE PIPELINE ON THE FULL TEST SET
# ============================================================


print("Validating pipeline against full test set...\n")

# Load raw test data (before feature engineering)
# We'll reconstruct from engineered features for this check
X_test_raw_check = pd.read_csv('../data/processed/X_test_unscaled.csv')
y_test_actual     = np.expm1(y_test)   # back to Lakhs

# Predict using the pipeline for each row
pipeline_preds = []

for _, row in X_test_raw_check.iterrows():
    # Reverse-engineer the original inputs from engineered features
    present_price = np.expm1(row['Present_Price_log'])
    kms_driven    = np.expm1(row['Kms_Driven_log'])
    car_age       = row['Car_Age']
    year          = 2024 - int(car_age)
    owner         = int(row['Owner'])

    # Reverse one-hot encoding
    if   row['Fuel_Diesel'] == 1: fuel_type = 'Diesel'
    elif row['Fuel_Petrol'] == 1: fuel_type = 'Petrol'
    else:                          fuel_type = 'CNG'

    seller_type  = 'Dealer'     if row['Seller_Type_enc']  == 1 else 'Individual'
    transmission = 'Manual'     if row['Transmission_enc'] == 1 else 'Automatic'

    pred = predict_car_price(
        present_price = present_price,
        kms_driven    = kms_driven,
        fuel_type     = fuel_type,
        seller_type   = seller_type,
        transmission  = transmission,
        owner         = owner,
        year          = year,
        model         = champion_model,
        scaler        = scaler,
        feature_cols  = feature_columns,
        needs_scaling = needs_scaling,
    )
    pipeline_preds.append(pred)

pipeline_preds = np.array(pipeline_preds)

# Compare to direct model predictions
if needs_scaling:
    direct_preds = np.expm1(champion_model.predict(X_test))
else:
    direct_preds = np.expm1(champion_model.predict(X_test_raw_check))

# Calculate agreement
differences = np.abs(pipeline_preds - direct_preds)
print(f"Pipeline vs Direct Predict Agreement:")
print(f"  Max difference:  ₹{differences.max():.4f}L")
print(f"  Mean difference: ₹{differences.mean():.6f}L")
print(f"  Agreement:       {'✅ PERFECT' if differences.max() < 0.01 else '⚠️ CHECK'}")

# Final pipeline R²
from sklearn.metrics import r2_score as r2
pipeline_r2   = r2(y_test_actual, pipeline_preds)
pipeline_rmse = np.sqrt(mean_squared_error(y_test_actual, pipeline_preds))
print(f"\nPipeline Test R²:   {pipeline_r2:.4f}")
print(f"Pipeline RMSE:      ₹{pipeline_rmse:.2f}L")

Validating pipeline against full test set...

Pipeline vs Direct Predict Agreement:
  Max difference:  ₹0.0050L
  Mean difference: ₹0.002744L
  Agreement:       ✅ PERFECT

Pipeline Test R²:   0.9443
Pipeline RMSE:      ₹1.71L


In [6]:
# SAVING THE COMPLETE MODEL PACKAGE

import hashlib

os.makedirs('../models', exist_ok=True)
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

# ── 1. Save champion model ────────────────────────────────────
model_path = '../models/champion_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(champion_model, f)

# Compute file hash for integrity verification
with open(model_path, 'rb') as f:
    model_hash = hashlib.md5(f.read()).hexdigest()

print(f"1. Saved: champion_model.pkl")
print(f"   Hash (MD5): {model_hash}")

# ── 2. Save scaler ────────────────────────────────────────────
scaler_path = '../models/scaler.pkl'
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f"\n2. Saved: scaler.pkl")

# ── 3. Save feature columns ───────────────────────────────────
features_path = '../models/feature_columns.pkl'
with open(features_path, 'wb') as f:
    pickle.dump(feature_columns, f)
print(f"\n3. Saved: feature_columns.pkl")
print(f"   Columns: {feature_columns}")

# ── 4. Save prediction pipeline function ─────────────────────
pipeline_path = '../models/predict_pipeline.pkl'
with open(pipeline_path, 'wb') as f:
    pickle.dump(predict_car_price, f)
print(f"\n4. Saved: predict_pipeline.pkl")

# ── 5. Save comprehensive metadata ───────────────────────────
metadata = {
    'project':          'Used Car Price Prediction',
    'version':          '1.0.0',
    'timestamp':        timestamp,
    'champion_name':    champion_meta['champion_name'],
    'model_type':       type(champion_model).__name__,
    'needs_scaling':    needs_scaling,

    'performance': {
        'test_r2':        round(pipeline_r2,   4),
        'test_rmse_lakhs':round(pipeline_rmse, 4),
        'test_mae_lakhs': round(
            float(np.mean(np.abs(y_test_actual - pipeline_preds))), 4),
    },

    'features': {
        'input_columns':    feature_columns,
        'n_features':       len(feature_columns),
        'target':           'Selling_Price_log',
        'target_transform': 'np.log1p(Selling_Price)',
        'reverse_transform':'np.expm1(prediction)',
    },

    'preprocessing': {
        'log_transformed':  ['Present_Price', 'Kms_Driven'],
        'label_encoded':    ['Seller_Type', 'Transmission'],
        'one_hot_encoded':  ['Fuel_Type'],
        'engineered':       ['Car_Age = current_year - Year'],
        'dropped':          ['Car_Name', 'Year'],
    },

    'valid_inputs': {
        'fuel_type':        ['Petrol', 'Diesel', 'CNG'],
        'seller_type':      ['Dealer', 'Individual'],
        'transmission':     ['Manual', 'Automatic'],
        'owner':            [0, 1, 2, 3],
        'year_range':       [2000, 2024],
        'present_price_range_lakhs': [0.5, 100.0],
        'kms_driven_range': [100, 500000],
    },

    'files': {
        'champion_model':   'models/champion_model.pkl',
        'scaler':           'models/scaler.pkl',
        'feature_columns':  'models/feature_columns.pkl',
        'pipeline':         'models/predict_pipeline.pkl',
        'metadata':         'models/model_metadata.json',
    },

    'integrity': {
        'model_md5': model_hash,
    }
}

meta_path = '../models/model_metadata.json'
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\n5. Saved: model_metadata.json")
print(f"\n{'='*50}")
print("MODEL PACKAGE COMPLETE")
print(f"{'='*50}")

1. Saved: champion_model.pkl
   Hash (MD5): 1941501a4ccb1241463c9f2aaae4abed

2. Saved: scaler.pkl

3. Saved: feature_columns.pkl
   Columns: ['Present_Price_log', 'Car_Age', 'Kms_Driven_log', 'Fuel_Diesel', 'Fuel_Petrol', 'Transmission_enc', 'Seller_Type_enc', 'Owner']

4. Saved: predict_pipeline.pkl

5. Saved: model_metadata.json

MODEL PACKAGE COMPLETE


In [7]:
# FINAL VERIFICATION — LOAD FROM SCRATCH AND PREDICT



print("="*50)
print("FINAL VERIFICATION — Loading fresh from disk")
print("="*50)

# Load everything from scratch
with open('../models/champion_model.pkl', 'rb') as f:
    v_model = pickle.load(f)

with open('../models/scaler.pkl', 'rb') as f:
    v_scaler = pickle.load(f)

with open('../models/feature_columns.pkl', 'rb') as f:
    v_features = pickle.load(f)

with open('../models/model_metadata.json', 'r') as f:
    v_meta = json.load(f)

v_needs_scaling = v_meta['needs_scaling']

print(f"\nLoaded model:    {v_meta['model_type']}")
print(f"Champion:        {v_meta['champion_name']}")
print(f"Version:         {v_meta['version']}")
print(f"Saved on:        {v_meta['timestamp']}")
print(f"Test R²:         {v_meta['performance']['test_r2']}")
print(f"RMSE (Lakhs):    ₹{v_meta['performance']['test_rmse_lakhs']}")

# Make a fresh prediction
print("\nFresh prediction test:")
print("-" * 40)

test_input = {
    'Car':           'Hyundai i20 2018 Petrol Manual',
    'present_price':  6.93,
    'kms_driven':     45000,
    'fuel_type':      'Petrol',
    'seller_type':    'Dealer',
    'transmission':   'Manual',
    'owner':          0,
    'year':           2018,
}

fresh_pred = predict_car_price(
    present_price = test_input['present_price'],
    kms_driven    = test_input['kms_driven'],
    fuel_type     = test_input['fuel_type'],
    seller_type   = test_input['seller_type'],
    transmission  = test_input['transmission'],
    owner         = test_input['owner'],
    year          = test_input['year'],
    model         = v_model,
    scaler        = v_scaler,
    feature_cols  = v_features,
    needs_scaling = v_needs_scaling,
)

print(f"Car:             {test_input['Car']}")
print(f"Showroom price:  ₹{test_input['present_price']} Lakhs")
print(f"Kilometres:      {test_input['kms_driven']:,} km")
print(f"Age:             {2024 - test_input['year']} years")
print(f"\n✅ Predicted selling price: ₹{fresh_pred} Lakhs")
print(f"\nModel package is fully working and deployment-ready!")

FINAL VERIFICATION — Loading fresh from disk

Loaded model:    Ridge
Champion:        Ridge Regression
Version:         1.0.0
Saved on:        20260524_201309
Test R²:         0.9443
RMSE (Lakhs):    ₹1.7107

Fresh prediction test:
----------------------------------------
Car:             Hyundai i20 2018 Petrol Manual
Showroom price:  ₹6.93 Lakhs
Kilometres:      45,000 km
Age:             6 years

✅ Predicted selling price: ₹5.64 Lakhs

Model package is fully working and deployment-ready!


In [8]:
print("="*50)
print("MODELS FOLDER CONTENTS")
print("="*50)

models_dir = '../models'
total_size  = 0

for filename in sorted(os.listdir(models_dir)):
    filepath  = os.path.join(models_dir, filename)
    size_kb   = os.path.getsize(filepath) / 1024
    size_mb   = size_kb / 1024
    total_size+= size_kb

    if size_mb >= 1:
        size_str = f"{size_mb:.2f} MB"
    else:
        size_str = f"{size_kb:.1f} KB"

    # File purpose
    purpose = {
        'champion_model.pkl':  'Trained champion ML model',
        'scaler.pkl':          'Fitted StandardScaler',
        'feature_columns.pkl': 'Feature column order list',
        'predict_pipeline.pkl':'Complete prediction function',
        'model_metadata.json': 'Metrics, version, config',
        'best_model.pkl':      'Alias for champion model',
        'champion_meta.json':  'Quick champion summary',
    }.get(filename, 'Supporting file')

    print(f"  {filename:<28} {size_str:>8}  — {purpose}")

print(f"\n  Total size: {total_size/1024:.2f} MB")
print(f"\n{'='*50}")
print("WHAT EACH FILE IS FOR")
print("="*50)
print("""
  champion_model.pkl   → loaded by the Streamlit app to predict
  scaler.pkl           → transforms user input before predicting
  feature_columns.pkl  → ensures correct feature order
  predict_pipeline.pkl → single function: raw input → price
  model_metadata.json  → app reads this for model info display
""")

MODELS FOLDER CONTENTS
  best_model.pkl                 0.8 KB  — Alias for champion model
  champion_meta.json             0.3 KB  — Quick champion summary
  champion_model.pkl             0.8 KB  — Trained champion ML model
  feature_columns.pkl            0.1 KB  — Feature column order list
  linear_regression_model.pkl    0.8 KB  — Supporting file
  model_metadata.json            1.8 KB  — Metrics, version, config
  predict_pipeline.pkl           0.0 KB  — Complete prediction function
  random_forest_model.pkl       1.14 MB  — Supporting file
  scaler.pkl                     0.9 KB  — Fitted StandardScaler
  xgboost_model.pkl            517.6 KB  — Supporting file

  Total size: 1.65 MB

WHAT EACH FILE IS FOR

  champion_model.pkl   → loaded by the Streamlit app to predict
  scaler.pkl           → transforms user input before predicting
  feature_columns.pkl  → ensures correct feature order
  predict_pipeline.pkl → single function: raw input → price
  model_metadata.json  → app rea

In [11]:
# ============================================================
# MODEL CARD — PROFESSIONAL DOCUMENTATION
# ============================================================

model_card = f"""
# Model Card — Used Car Price Predictor

## Model Details
- **Name:** {v_meta['champion_name']}
- **Type:** {v_meta['model_type']}
- **Version:** {v_meta['version']}
- **Created:** {v_meta['timestamp']}
- **Author:** Stephen Mutu 

## Intended Use
- Predict the fair market selling price of used cars in India.

## Training Data
- Source: CarDekho Vehicle Dataset (Kaggle)
- Size: 301 rows
- Features: {len(v_features)} engineered features

## Performance

| Metric | Value |
|--------|-------|
| Test R² | {v_meta['performance']['test_r2']} |
| Test RMSE | ₹{v_meta['performance']['test_rmse_lakhs']} Lakhs |
| Test MAE | ₹{v_meta['performance']['test_mae_lakhs']} Lakhs |

## Features Used

{chr(10).join(f'- {f}' for f in v_features)}

## Preprocessing Steps
1. Log-transform numerical columns
2. Feature engineering
3. Encoding categorical variables
4. Scaling selected features

## Limitations
- Small dataset
- India-only market
- No accident-history data

## Ethical Considerations
- Predictions are estimates only
- Users should compare multiple valuations

## Example Usage

    import pickle
    import json

    with open('models/champion_model.pkl', 'rb') as f:
        model = pickle.load(f)

    print("Model loaded successfully")
"""

# ============================================================
# SAVE MODEL CARD
# ============================================================

with open('../models/MODEL_CARD.md', 'w', encoding='utf-8') as f:
    f.write(model_card)

print(model_card)
print("\nModel card saved successfully!")


# Model Card — Used Car Price Predictor

## Model Details
- **Name:** Ridge Regression
- **Type:** Ridge
- **Version:** 1.0.0
- **Created:** 20260524_201309
- **Author:** [Your Name]

## Intended Use
- Predict the fair market selling price of used cars in India.

## Training Data
- Source: CarDekho Vehicle Dataset (Kaggle)
- Size: 301 rows
- Features: 8 engineered features

## Performance

| Metric | Value |
|--------|-------|
| Test R² | 0.9443 |
| Test RMSE | ₹1.7107 Lakhs |
| Test MAE | ₹0.9212 Lakhs |

## Features Used

- Present_Price_log
- Car_Age
- Kms_Driven_log
- Fuel_Diesel
- Fuel_Petrol
- Transmission_enc
- Seller_Type_enc
- Owner

## Preprocessing Steps
1. Log-transform numerical columns
2. Feature engineering
3. Encoding categorical variables
4. Scaling selected features

## Limitations
- Small dataset
- India-only market
- No accident-history data

## Ethical Considerations
- Predictions are estimates only
- Users should compare multiple valuations

## Example Usage

   